# Séance 4 : regrouper et combiner

Objectif : passer de 2 304 relevés à des réponses courtes (« quel est le prix moyen du riz dans chaque département ? »), puis enrichir les relevés avec d'autres tableaux (population, catégorie des produits).

Règle du jour : on regarde la forme du résultat (`shape`, `head`) après chaque `groupby` et **on compte les lignes avant et après chaque jointure**.

In [1]:
import pandas as pd

prix = pd.read_csv("donnees/prix_marches.csv")
departements = pd.read_csv("donnees/departements.csv")
produits = pd.read_csv("donnees/produits.csv")
prix.shape, departements.shape, produits.shape

((2304, 6), (10, 4), (8, 4))

## 1. Compter avec `value_counts`

In [2]:
prix["produit"].value_counts()

produit
Riz importé       288
Riz local         288
Maïs moulu        288
Farine de blé     288
Haricot noir      288
Huile végétale    288
Sucre             288
Spaghetti         288
Name: count, dtype: int64

In [3]:
prix["departement"].value_counts().head(3)

departement
Ouest         384
Artibonite    384
Nord          192
Name: count, dtype: int64

L'Ouest et l'Artibonite ont deux marchés chacun, d'où deux fois plus de relevés. `normalize=True` donne des proportions :

In [4]:
prix["departement"].value_counts(normalize=True).round(3).head(3)

departement
Ouest         0.167
Artibonite    0.167
Nord          0.083
Name: proportion, dtype: float64

## 2. Diviser, appliquer, combiner : `groupby`

`prix.groupby("produit")["prix_htg"].mean()` se lit en trois temps :

1. **diviser** les lignes en paquets, un par produit ;
2. **appliquer** `mean` à la colonne `prix_htg` de chaque paquet ;
3. **combiner** les résultats dans une Series, indexée par produit.

In [5]:
prix.groupby("produit")["prix_htg"].mean().round(1)

produit
Farine de blé      553.1
Haricot noir      1020.4
Huile végétale    1960.5
Maïs moulu         444.4
Riz importé        704.9
Riz local          860.6
Spaghetti          101.3
Sucre              621.3
Name: prix_htg, dtype: float64

In [6]:
prix.groupby("produit")["prix_htg"].mean().round(1).sort_values(ascending=False).head(3)

produit
Huile végétale    1960.5
Haricot noir      1020.4
Riz local          860.6
Name: prix_htg, dtype: float64

Sans choisir de colonne, pandas essaie de calculer la moyenne de **toutes** les colonnes, y compris le texte :

In [7]:
prix.groupby("produit").mean()

TypeError: dtype 'str' does not support operation 'mean'

## 3. Plusieurs calculs : `agg`

Une liste de fonctions donne une colonne par fonction. Avec des **paires nommées** `nom=(colonne, fonction)`, on choisit le nom des colonnes du résultat.

In [8]:
prix.groupby("produit")["prix_htg"].agg(["mean", "min", "max"]).round(1)

,mean,min,max
produit,,,
Farine de blé,553.1,365.0,813.0
Haricot noir,1020.4,681.0,1368.0
Huile végétale,1960.5,1366.0,2884.0
Maïs moulu,444.4,297.0,606.0
Riz importé,704.9,480.0,1011.0
Riz local,860.6,608.0,1186.0
Spaghetti,101.3,69.0,150.0
Sucre,621.3,413.0,914.0


In [9]:
resume = prix.groupby("produit").agg(
    releves=("prix_htg", "size"),
    moyen=("prix_htg", "mean"),
    plus_cher=("prix_htg", "max"),
).round(1)
resume

,releves,moyen,plus_cher
produit,,,
Farine de blé,288,553.1,813.0
Haricot noir,288,1020.4,1368.0
Huile végétale,288,1960.5,2884.0
Maïs moulu,288,444.4,606.0
Riz importé,288,704.9,1011.0
Riz local,288,860.6,1186.0
Spaghetti,288,101.3,150.0
Sucre,288,621.3,914.0


### `count` ou `size` ?

`size` compte les lignes du groupe ; `count` compte les valeurs **non manquantes** d'une colonne. Sur un fichier propre, c'est pareil. Sur le fichier brut de la séance 3, non :

In [10]:
brut = pd.read_csv("donnees/prix_marches_brut.csv")
brut["produit"] = brut["produit"].str.strip().str.capitalize()
brut.groupby("produit")["prix_htg"].agg(["size", "count"]).head(3)

,size,count
produit,,
Farine de blé,292,281
Haricot noir,292,284
Huile végétale,291,282


## 4. Plusieurs clés et `reset_index`

Avec une liste de clés, l'index du résultat a deux niveaux. `reset_index()` remet les clés en colonnes : on retrouve un tableau ordinaire, qu'on peut filtrer, trier, joindre.

In [11]:
par_dep = prix.groupby(["produit", "departement"])["prix_htg"].mean().round(1)
par_dep.head(4)

produit        departement
Farine de blé  Artibonite     516.3
               Centre         536.4
               Grand'Anse     617.7
               Nippes         565.6
Name: prix_htg, dtype: float64

In [12]:
par_dep = par_dep.reset_index()
par_dep.head(4)

,produit,departement,prix_htg
0,Farine de blé,Artibonite,516.3
1,Farine de blé,Centre,536.4
2,Farine de blé,Grand'Anse,617.7
3,Farine de blé,Nippes,565.6


**Exercice type** : le département le plus cher pour chaque produit. On trie du plus cher au moins cher, puis on garde la première ligne de chaque produit.

In [13]:
plus_cher = par_dep.sort_values("prix_htg", ascending=False).drop_duplicates("produit")
plus_cher.sort_values("produit")

,produit,departement,prix_htg
2,Farine de blé,Grand'Anse,617.7
12,Haricot noir,Grand'Anse,1108.0
22,Huile végétale,Grand'Anse,2209.8
32,Maïs moulu,Grand'Anse,481.2
42,Riz importé,Grand'Anse,787.8
52,Riz local,Grand'Anse,937.0
62,Spaghetti,Grand'Anse,114.2
72,Sucre,Grand'Anse,698.6


Grand'Anse est le département le plus cher pour les 8 produits : c'est le plus loin de Port-au-Prince, où arrivent les produits importés.

## 5. Tableau croisé : `pivot_table`

`pivot_table` regroupe selon deux clés et met l'une en lignes, l'autre en colonnes. Ici, le prix moyen de chaque produit par année.

In [14]:
prix["annee"] = prix["mois"].str[:4]
par_annee = prix.pivot_table(index="produit", columns="annee", values="prix_htg", aggfunc="mean").round(1)
par_annee

annee,2024,2025
produit,,
Farine de blé,476.0,630.2
Haricot noir,886.8,1153.9
Huile végétale,1691.4,2229.6
Maïs moulu,387.1,501.6
Riz importé,610.2,799.5
Riz local,753.3,968.0
Spaghetti,87.7,115.0
Sucre,537.2,705.4


In [15]:
par_annee["hausse_%"] = ((par_annee["2025"] / par_annee["2024"] - 1) * 100).round(1)
par_annee.sort_values("hausse_%", ascending=False).head(3)

annee,2024,2025,hausse_%
produit,,,
Farine de blé,476.0,630.2,32.4
Huile végétale,1691.4,2229.6,31.8
Sucre,537.2,705.4,31.3


## 6. Joindre deux tableaux : `merge`

`prix` connaît le département de chaque relevé, `departements` connaît la population de chaque département. `merge` les met côte à côte grâce à la colonne commune, la **clé**.

In [16]:
avec_pop = prix.merge(departements, on="departement", how="left")
prix.shape, avec_pop.shape

((2304, 7), (2304, 10))

In [17]:
avec_pop[["mois", "departement", "produit", "prix_htg", "chef_lieu", "population"]].head(3)

,mois,departement,produit,prix_htg,chef_lieu,population
0,2024-01,Ouest,Riz importé,488.0,Port-au-Prince,4029705
1,2024-01,Ouest,Riz local,623.0,Port-au-Prince,4029705
2,2024-01,Ouest,Maïs moulu,297.0,Port-au-Prince,4029705


Même nombre de lignes, quatre colonnes de plus : chaque relevé a trouvé son département. Et avec la catégorie des produits :

In [18]:
complet = avec_pop.merge(produits[["produit", "categorie", "importe"]], on="produit", how="left")
complet.shape

(2304, 12)

In [19]:
complet.groupby("importe")["prix_htg"].mean().round(1)

importe
False    775.1
True     788.2
Name: prix_htg, dtype: float64

### `inner` ou `left` ?

- `how="inner"` : seulement les lignes dont la clé existe **des deux côtés**.
- `how="left"` : toutes les lignes de gauche ; `NaN` quand la clé manque à droite.

Un petit tableau de populations où un département est écrit autrement :

In [20]:
pop_mal_ecrite = pd.DataFrame({"departement": ["Ouest", "Nord", "Grande-Anse"],
                               "population": [4029705, 1067177, 468301]})
releves = prix[prix["departement"].isin(["Ouest", "Nord", "Grand'Anse"])]
releves.shape

(768, 7)

In [21]:
releves.merge(pop_mal_ecrite, on="departement", how="inner").shape

(576, 8)

In [22]:
gauche = releves.merge(pop_mal_ecrite, on="departement", how="left")
gauche.shape, int(gauche["population"].isna().sum())

((768, 8), 192)

Avec `inner`, les 192 relevés de Grand'Anse disparaissent **sans message d'erreur**. Avec `left`, ils restent, population manquante : on voit le problème et on le répare (séance 3).

### Une jointure peut aussi **ajouter** des lignes

Si la clé apparaît deux fois dans le tableau de droite, chaque ligne de gauche est recopiée deux fois.

In [23]:
produits_doubles = pd.concat([produits, produits[produits["produit"] == "Sucre"]])
prix.merge(produits_doubles, on="produit", how="left").shape

(2592, 10)

2 592 lignes au lieu de 2 304 : les 288 relevés de sucre sont en double. D'où la règle : **compter avant et après**.

## 7. Une moyenne pondérée par la population

Le prix moyen national du riz importé « par habitant » : chaque département compte selon sa population, pas selon son nombre de marchés.

In [24]:
riz = complet[complet["produit"] == "Riz importé"]
par_departement = riz.groupby("departement").agg(prix=("prix_htg", "mean"), population=("population", "first"))
simple = par_departement["prix"].mean()
pondere = (par_departement["prix"] * par_departement["population"]).sum() / par_departement["population"].sum()
round(float(simple), 1), round(float(pondere), 1)

(713.8, 691.7)

La moyenne pondérée est plus basse : l'Ouest, où le riz importé est moins cher (667 gourdes), pèse 4 millions d'habitants sur 11.

## 8. Empiler : `concat`

`merge` met des colonnes côte à côte ; `concat` **empile** des lignes qui ont les mêmes colonnes, par exemple deux années reçues dans deux fichiers.

In [25]:
releves_2024 = prix[prix["annee"] == "2024"]
releves_2025 = prix[prix["annee"] == "2025"]
deux_ans = pd.concat([releves_2024, releves_2025])
releves_2024.shape, releves_2025.shape, deux_ans.shape

((1152, 7), (1152, 7), (2304, 7))

In [26]:
deux_ans.index[1150:1154]

RangeIndex(start=1150, stop=1154, step=1)

In [27]:
pd.concat([releves_2024, releves_2025], ignore_index=True).index[-1]

2303

Sans `ignore_index=True`, l'index garde les numéros des tableaux d'origine. Ici ils se suivent par chance ; avec deux fichiers lus séparément, chacun commencerait à 0 et l'index aurait des doublons.

## 9. Enregistrer le résultat : `to_csv`

In [28]:
resume.reset_index().to_csv("resume_produits.csv", index=False)
pd.read_csv("resume_produits.csv").head(3)

,produit,releves,moyen,plus_cher
0,Farine de blé,288,553.1,813.0
1,Haricot noir,288,1020.4,1368.0
2,Huile végétale,288,1960.5,2884.0


Sans `index=False`, la colonne d'index est écrite dans le fichier et revient à la relecture comme une colonne `Unnamed: 0`.

## Exercices en binôme

Les énoncés et les résultats attendus sont sur la page de la séance 4.

## Aller plus loin

- **`transform` contre `agg`** : `prix.groupby("produit")["prix_htg"].transform("mean")` a combien de lignes ? Pourquoi est-ce utile pour comparer chaque relevé à la moyenne de son produit ?
- **`pd.crosstab`** : compter les relevés par département et par produit en une ligne.
- **`melt`** : l'inverse de `pivot_table`, pour repasser d'un tableau large à un tableau long.
- **`validate=` et `indicator=`** de `merge` : faire échouer une jointure qui duplique des lignes, et voir d'où vient chaque ligne.
- **`pd.read_excel`** : lire un fichier Excel (il faut installer `openpyxl`).

## Sources

- pandas, « Group by: split-apply-combine » : https://pandas.pydata.org/docs/user_guide/groupby.html
- pandas, « Reshaping and pivot tables » : https://pandas.pydata.org/docs/user_guide/reshaping.html
- pandas, « Merge, join, concatenate and compare » : https://pandas.pydata.org/docs/user_guide/merging.html
- Wes McKinney, *Python for Data Analysis*, chapitres 8 et 10 : https://wesmckinney.com/book/